### TTC Weekly Disruption Prediction Pipeline

In [1]:
import requests
import pandas as pd
from datetime import datetime
from pathlib import Path
import joblib
import numpy as np

In [2]:
# Toronto coordinates (weather forecast call)
TORONTO_LAT = 43.6532
TORONTO_LON = -79.3832

# GTFS raw files location
# GTFS_DIR = Path("data/raw") 
OUTPUT_DIR = Path("output")
MODEL_DIR = Path("models")

ttc_trip_df = pd.read_csv(OUTPUT_DIR / "TTC_trip_details.csv")
location_df = pd.read_csv(OUTPUT_DIR / "unique_locations_lookup.csv")
route_location_df = pd.read_csv(OUTPUT_DIR / "route_location_lookup.csv")

# 50 selected TTC bus route short_names
SELECTED_ROUTES = [7, 11, 12, 24, 25, 29, 32, 34, 35, 36, 37, 38, 39, 40, 41, 43, 45, 47, 52, 53, 54, 59, 60, 
                   63, 68, 72, 73, 79, 84, 85, 86, 89, 94, 95, 96, 97, 100, 102, 108, 110, 116, 123, 129, 133, 165, 924, 925, 929, 939, 960]

# match types
ttc_trip_df["route_short_name"] = (pd.to_numeric(ttc_trip_df["route_short_name"], errors="coerce"))
route_location_df["route"] = (pd.to_numeric(route_location_df["route"], errors="coerce"))
route_location_df["location_id"] = (pd.to_numeric(route_location_df["location_id"], errors="coerce"))

In [3]:
# Fetches an HOURLY 3-day weather forecast for Toronto using the Open-Meteo API
# Returns a DataFrame with one row per HOUR
def get_weather_forecast():                     
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": TORONTO_LAT,
        "longitude": TORONTO_LON,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "wind_speed_10m",
            "visibility",
        ],
        "wind_speed_unit": "kmh",
        "timezone": "America/Toronto",
        "forecast_days": 3,
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    hourly = data["hourly"]
    weather_df = pd.DataFrame({
        "datetime": pd.to_datetime(hourly["time"]),
        "temp_celsius": hourly["temperature_2m"],
        "humidity_pct": hourly["relative_humidity_2m"],
        "precipitation_mm": hourly["precipitation"],
        "wind_speed_kmh": hourly["wind_speed_10m"],
        "visibility_km": [v / 1000 for v in hourly["visibility"]],  # Open-Meteo returns meters
    })

    weather_df["date"] = weather_df["datetime"].dt.date
    weather_df["hour"] = weather_df["datetime"].dt.hour

    return weather_df

<div style="background-color:#d2f0c5; font-size:12.8px; padding:5px">
    
##### Create future prediction features
Builds the future prediction feature matrix, using the SAME feature engineering logic used during training (time-based flags, weather flags, weather_intensity score)

Parameters
* gtfs : Output of load_gtfs() -- columns: route_short_name, direction_id, scheduled_hour. One row per (route, hour) combination the route
        actually runs
* weather : hourly forecast with date, hour, temp_celsius, humidity_pct, precipitation_mm, wind_speed_kmh, visibility_km.
* location_lookup : Two columns: route, location_id. Historical typical/most-frequent location_id per route 
* ttc_trip_Detauls : route, dir_E, dir_N, dir_S, dir_W, dir_UNKNOWN
* n_days : Number of future days to generate predictions for, starting today.
</div>

In [4]:
FEATURE_ORDER = ["route", "location_id", "year", "month", "hour", "day_of_week",
    "is_weekend", "is_rush_hour", "temp_celsius", "humidity_pct", "precipitation_mm",
    "wind_speed_kmh", "visibility_km", "is_rainy", "is_heavy_rain", "is_cold", "is_hot",
    "low_visibility", "high_wind", "weather_intensity", "dir_E", "dir_N", "dir_S", "dir_UNKNOWN", "dir_W",]

# Create prediction feature dataframe
def create_features(ttc_trip_df, weather_df, route_location_df, n_days=3):

    # Route + operating hour + direction
    route_hours = (ttc_trip_df[["route_short_name", "scheduled_hour", "dir_E", "dir_N", "dir_S", "dir_W",
                "dir_UNKNOWN"]].drop_duplicates().rename(columns={"route_short_name":"route", "scheduled_hour":"hour"}))

    route_hours["route"] = (route_hours["route"].astype(int))

    # Add route-location combinations
    grid = route_hours.merge(route_location_df, on="route", how="left")
    if grid["location_id"].isna().any():
        missing = (
            grid.loc[grid["location_id"].isna(), "route"].unique()
        )

        print("Missing location routes:", missing)

    # Create future dates
    today = (pd.Timestamp.now(tz="America/Toronto").normalize())
    dates = pd.DataFrame({"date":pd.date_range(today, periods=n_days, freq="D").date})
    grid["_key"] = 1
    dates["_key"] = 1
    grid = (grid.merge( dates, on="_key").drop(columns="_key"))

    # Merge weather
    weather_merge = weather_df.copy()
    weather_merge["date"] = (pd.to_datetime(weather_merge["date"]).dt.date)
    grid = grid.merge(weather_merge[["date","hour","temp_celsius","humidity_pct",
                "precipitation_mm","wind_speed_kmh","visibility_km"]],
        on=[
            "date", "hour"
        ],
        how="left"
    )

    if grid["temp_celsius"].isna().sum() > 0:
        print("Missing weather rows:", grid["temp_celsius"].isna().sum())

    # Time features
    grid["datetime"] = (pd.to_datetime(grid["date"]) + pd.to_timedelta(
            grid["hour"], unit="h"))

    grid["year"] = (grid["datetime"].dt.year)
    grid["month"] = (grid["datetime"].dt.month)
    grid["day_of_week"] = (grid["datetime"].dt.dayofweek)
    grid["is_weekend"] = (grid["day_of_week"].isin([5,6]).astype(int))
    grid["is_rush_hour"] = (grid["hour"].isin([7, 8, 9, 16, 17, 18]).astype(int))

    # Weather labels
    grid["is_rainy"] = (grid["precipitation_mm"] > 0).astype(int)
    grid["is_heavy_rain"] = (grid["precipitation_mm"] > 5).astype(int)
    grid["is_cold"] = (grid["temp_celsius"] < 0).astype(int)
    grid["is_hot"] = (grid["temp_celsius"] > 25).astype(int)
    grid["low_visibility"] = (grid["visibility_km"] < 5).astype(int)
    grid["high_wind"] = (grid["wind_speed_kmh"] > 30).astype(int)
    grid["weather_intensity"] = (grid["precipitation_mm"] * 0.5 + (grid["wind_speed_kmh"] / 20) +
        grid["is_rainy"] + grid["is_cold"])

    # Ensure direction columns are numeric
    dir_cols = ["dir_E", "dir_N", "dir_S", "dir_W", "dir_UNKNOWN"]
    for c in dir_cols:
        if c not in grid.columns:
            grid[c] = 0
            
        grid[c] = (grid[c].astype(int))

    # Final ML dataset
    missing = [
        c for c in FEATURE_ORDER
        if c not in grid.columns
    ]

    if missing:
        raise ValueError(
            f"Missing features: {missing}"
        )

    X_future = (grid[FEATURE_ORDER].copy())
    meta = (grid[["route", "location_id", "date", "hour"]].copy())

    return X_future, meta

In [5]:
weather = get_weather_forecast()
X_future, meta = create_features(ttc_trip_df, weather, route_location_df, n_days=3)

print("Feature shape:", X_future.shape)

Feature shape: (2890851, 25)


In [10]:
# X_future.describe()

,route,location_id,year,month,hour,day_of_week,is_weekend,is_rush_hour,temp_celsius,humidity_pct,...,is_cold,is_hot,low_visibility,high_wind,weather_intensity,dir_E,dir_N,dir_S,dir_UNKNOWN,dir_W
count,2.890851e+06,2.890851e+06,2890851.0,2890851.0,2.890851e+06,2.890851e+06,2890851.0,2.890851e+06,2.890851e+06,2.890851e+06,...,2890851.0,2.890851e+06,2.890851e+06,2890851.0,2.890851e+06,2.890851e+06,2.890851e+06,2.890851e+06,2890851.0,2.890851e+06
mean,1.219518e+02,1.456427e+04,2026.0,7.0,1.236838e+01,3.000000e+00,0.0,2.782039e-01,2.416464e+01,5.856634e+01,...,0.0,4.740767e-01,1.592213e-01,0.0,5.074718e-01,2.517857e-01,2.489641e-01,2.471636e-01,0.0,2.520867e-01
std,2.184800e+02,7.810393e+03,0.0,0.0,6.646311e+00,8.164967e-01,0.0,4.481144e-01,3.002052e+00,1.488222e+01,...,0.0,4.993276e-01,3.658824e-01,0.0,2.414985e-01,4.340389e-01,4.324130e-01,4.313627e-01,0.0,4.342108e-01
min,7.000000e+00,1.000000e+00,2026.0,7.0,0.000000e+00,2.000000e+00,0.0,0.000000e+00,1.670000e+01,2.500000e+01,...,0.0,0.000000e+00,0.000000e+00,0.0,1.200000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00
25%,3.700000e+01,8.542000e+03,2026.0,7.0,7.000000e+00,2.000000e+00,0.0,0.000000e+00,2.210000e+01,5.000000e+01,...,0.0,0.000000e+00,0.000000e+00,0.0,3.550000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00
50%,6.000000e+01,1.421200e+04,2026.0,7.0,1.300000e+01,3.000000e+00,0.0,0.000000e+00,2.490000e+01,6.000000e+01,...,0.0,0.000000e+00,0.000000e+00,0.0,5.100000e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00
75%,9.600000e+01,2.155900e+04,2026.0,7.0,1.800000e+01,4.000000e+00,0.0,1.000000e+00,2.650000e+01,6.700000e+01,...,0.0,1.000000e+00,0.000000e+00,0.0,6.000000e-01,1.000000e+00,0.000000e+00,0.000000e+00,0.0,1.000000e+00
max,9.600000e+02,2.772100e+04,2026.0,7.0,2.300000e+01,4.000000e+00,0.0,1.000000e+00,2.970000e+01,8.900000e+01,...,0.0,1.000000e+00,1.000000e+00,0.0,1.340000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.0,1.000000e+00


In [11]:
# Null value check
# X_future.isna().sum().sum()

np.int64(0)

### Model 

In [9]:
# Load trained models
major_delay_model = joblib.load(MODEL_DIR / "xgb_major_delay.pkl")
gapping_model = joblib.load(MODEL_DIR / "xgb_gapping.pkl")
bunching_model = joblib.load(MODEL_DIR / "xgb_bunching.pkl")

# Predict probabilities
print("Predicting major delay...")
major_delay_prob = major_delay_model.predict_proba(X_future)[:, 1]
print("Predicting gapping...")
gapping_prob = gapping_model.predict_proba(X_future)[:, 1]
print("Predicting bunching...")
bunching_prob = bunching_model.predict_proba(X_future)[:, 1]

# Binary predictions (from threshold)
MAJOR_DELAY_THRESHOLD = 0.552
GAPPING_THRESHOLD = 0.603
BUNCHING_THRESHOLD = 0.591
major_delay_pred = (major_delay_prob >= MAJOR_DELAY_THRESHOLD).astype(int)
gapping_pred = (gapping_prob >= GAPPING_THRESHOLD).astype(int)
bunching_pred = (bunching_prob >= BUNCHING_THRESHOLD).astype(int)

# Build prediction dataframe
predictions = meta.copy()
predictions["major_delay_probability"] = major_delay_prob
# predictions["major_delay_prediction"] = major_delay_pred
predictions["gapping_probability"] = gapping_prob
# predictions["gapping_prediction"] = gapping_pred
predictions["bunching_probability"] = bunching_prob
# predictions["bunching_prediction"] = bunching_pred

# Add direction columns
direction_cols = X_future[["dir_E", "dir_N", "dir_S", "dir_W", "dir_UNKNOWN"]].reset_index(drop=True)
predictions = pd.concat([predictions.reset_index(drop=True), direction_cols], axis=1)

# Vectorized direction assignment (FAST)
predictions["direction"] = np.select([predictions["dir_E"] == 1, predictions["dir_W"] == 1,
        predictions["dir_N"] == 1, predictions["dir_S"] == 1], ["East", "West", "North", "South"], default="Unknown")

# Remove one-hot columns
predictions = predictions.drop(columns=["dir_E", "dir_N", "dir_S", "dir_W", "dir_UNKNOWN"])
# Sort
predictions = predictions.sort_values(["date", "hour", "route", "location_id"]).reset_index(drop=True)

# Save
predictions.to_csv(OUTPUT_DIR / "predicted_dataset.parquet", index=False, compression='gzip')
# predictions.to_csv(OUTPUT_DIR / "predicted_dataset.csv", index=False)
print(f"Saved {len(predictions):,} predictions.")

Predicting major delay...
Predicting gapping...
Predicting bunching...
Saved 2,890,851 predictions.
